In [67]:
# Imports numPy (math library), pandas (data analysis library), nflreadpy (NFL data library), 
# and scikit-learn (machine learning library).
import numpy as np
import pandas as pd
import nflreadpy as nfl

# Imports the Pipeline class, StandardScaler class, LogisticRegression class, 
# accuracy_score function, and log_loss function from scikit-learn.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

# Loads NFL game schedules for the seasons 2015 to 2025, converts the "gameday" column 
# to datetime format, sorts the games by date, and creates a new column "home_win" that
#  indicates whether the home team won (1) or lost (0).
games = nfl.load_schedules(seasons=range(2015, 2026)).to_pandas()
games["gameday"] = pd.to_datetime(games["gameday"])
games = games.sort_values("gameday").reset_index(drop=True)
games["home_win"] = (games["home_score"] > games["away_score"]).astype(int)

teams = sorted(set(games["home_team"]) | set(games["away_team"]))
print(len(teams), teams)

35 ['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'OAK', 'PHI', 'PIT', 'SD', 'SEA', 'SF', 'STL', 'TB', 'TEN', 'WAS']


In [68]:
# Creates a dictonary called team_fix, which maps the old team names to the new team names.
team_fix = {"OAK": "LV", "SD": "LAC", "STL": "LA"}

# Goes through the columns and swaps every old name for its new one
games["home_team"] = games["home_team"].replace(team_fix)
games["away_team"] = games["away_team"].replace(team_fix)

# Recount the teams
teams = sorted(set(games["home_team"]) | set(games["away_team"]))
print(len(teams), teams)

32 ['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']


In [69]:
# Takes each column from one team's point of view and renames them. 
home = games[["game_id", "gameday", "home_team", "home_score", "away_score"]].copy()
home.columns = ["game_id", "gameday", "team", "pts_for", "pts_against"]
home["is_home"] = 1

# Takes each column from the other team's point of view and renames them.
away = games[["game_id", "gameday", "away_team", "away_score", "home_score"]].copy()
away.columns = ["game_id", "gameday", "team", "pts_for", "pts_against"]
away["is_home"] = 0

# Stacks the two tables and sorts them by team and date. 
# Then, it creates two new columns: "win" indicates whether the team won (1) or lost (0), 
# and "margin" calculates the point difference between the team's score and the opponent's score.
team_games = pd.concat([home, away]).sort_values(["team", "gameday"]).reset_index(drop=True)
team_games["win"] = (team_games["pts_for"] > team_games["pts_against"]).astype(int)
team_games["margin"] = team_games["pts_for"] - team_games["pts_against"]
print(team_games.shape)

(6056, 8)


In [70]:
# Dictionary that maps each column we want to average to the name its result gets.
rolling_names = {
    "win": "win_pct_last5",
    "margin": "margin_last5",
    "pts_for": "pts_for_last5",
    "pts_against": "pts_against_last5",
}

# Makes each team its own group.
g = team_games.groupby("team")

# Loops once per dictionary entry. Items() gives the key and value together.
for col, new_name in rolling_names.items():
    team_games[new_name] = g[col].transform(
        lambda s: s.shift(1).rolling(5, min_periods=1).mean())

team_games[team_games["team"] == "KC"].head(6)

,game_id,gameday,team,pts_for,pts_against,is_home,win,margin,win_pct_last5,margin_last5,pts_for_last5,pts_against_last5
2819,2015_01_KC_HOU,2015-09-13,KC,27,20,0,1,7,NaN,NaN,NaN,NaN
2820,2015_02_DEN_KC,2015-09-17,KC,24,31,1,0,-7,1.000000,7.000000,27.000000,20.000000
2821,2015_03_KC_GB,2015-09-28,KC,28,38,0,0,-10,0.500000,0.000000,25.500000,25.500000
2822,2015_04_KC_CIN,2015-10-04,KC,21,36,0,0,-15,0.333333,-3.333333,26.333333,29.666667
2823,2015_05_CHI_KC,2015-10-11,KC,17,18,1,0,-1,0.250000,-6.250000,25.000000,31.250000
2824,2015_06_KC_MIN,2015-10-18,KC,10,16,0,0,-6,0.200000,-5.200000,23.400000,28.600000


In [71]:
# Creates a list with the dictionary's four new names
feat_cols = list(rolling_names.values())
# Keeps only the game Id, home/away flag, and four features
feats = team_games[["game_id", "is_home"] + feat_cols]

# Split feats into home teams and away teams, drop the flag, and rename the features.
home_f = feats[feats["is_home"] == 1].drop(columns="is_home")
home_f = home_f.rename(columns={c: f"home_{c}" for c in feat_cols})

away_f = feats[feats["is_home"] == 0].drop(columns="is_home")
away_f = away_f.rename(columns={c: f"away_{c}" for c in feat_cols})

# Attach both team's features to each game by game_id
games = games.merge(home_f, on="game_id").merge(away_f, on="game_id")

# Creates new columns that calculate the difference between the home and away team's features.
for c in feat_cols:
    games[c.replace("_last5", "_diff")] = games[f"home_{c}"] - games[f"away_{c}"]

print(games.shape)
print([c for c in games.columns if c.endswith("_diff")])

(3028, 59)
['win_pct_diff', 'margin_diff', 'pts_for_diff', 'pts_against_diff']


In [72]:
# Settings. Start is the beginning rating, K controls how fast the rating move, and 
# HOME_ADV gives the home team a bonus in the win-probablilty formula.
START = 1500
K = 20
HOME_ADV = 55

# Creates a function that adds the bonus to the home team's elo rating, and then puts it in the formula
# that turns a gap into a probability between 0 and 1.
def expected_home_win(home_elo, away_elo):
    diff = (home_elo + HOME_ADV) - away_elo
    return 1 / (1 + 10 ** (-diff / 400))

# elo is an empty dictionary, and home_elos and away_elos are empty lists.
elo = {}
home_elos, away_elos = [], []
current_season = None

# Loops through each row in the games DataFrame. If the season changes, it decays all team 
# ratings towards the starting rating. Then, it calculates the expected outcome of the game 
# based on the current ratings, updates the ratings based on the actual outcome, and appends the 
# current ratings to the home_elos and away_elos lists.
for row in games.itertuples():
    if row.season != current_season:
        for t in elo:
            elo[t] = elo[t] * (2 / 3) + START * (1 / 3)
        current_season = row.season

    h = elo.get(row.home_team, START)
    a = elo.get(row.away_team, START)
    home_elos.append(h)
    away_elos.append(a)

    if row.home_score > row.away_score:
        actual = 1.0
    elif row.home_score < row.away_score:
        actual = 0.0
    else:
        actual = 0.5

    expected = expected_home_win(h, a)
    margin = abs(row.home_score - row.away_score)
    change = K * np.log(margin + 1) * (actual - expected)

    elo[row.home_team] = h + change
    elo[row.away_team] = a - change

games["home_elo"] = home_elos
games["away_elo"] = away_elos
games["elo_diff"] = games["home_elo"] - games["away_elo"]

ratings = pd.Series(elo).sort_values(ascending=False)
print(ratings.head(3).round(0))
print(ratings.tail(3).round(0))
print(round(ratings.mean(), 1))

SEA    1768.0
LA     1671.0
BUF    1669.0
dtype: float64
NYJ    1277.0
TEN    1262.0
LV     1261.0
dtype: float64
1500.0


In [73]:
games["rest_diff"] = games["home_rest"] - games["away_rest"]

v1_cols = ["win_pct_diff", "margin_diff", "rest_diff"]
v2_cols = v1_cols + ["pts_for_diff", "pts_against_diff", "elo_diff"]

games = games.dropna(subset=v2_cols).reset_index(drop=True)
print(len(games))
print(games[v2_cols].isna().sum().sum())
print(games[v2_cols].describe().round(2))

3012
0
       win_pct_diff  margin_diff  rest_diff  pts_for_diff  pts_against_diff  \
count       3012.00      3012.00    3012.00       3012.00           3012.00   
mean          -0.00        -0.25       0.01         -0.11              0.14   
std            0.37        10.79       2.50          7.77              6.81   
min           -1.00       -49.00      -8.00        -32.00            -22.00   
25%           -0.20        -7.40       0.00         -5.40             -4.40   
50%            0.00        -0.20       0.00          0.00              0.00   
75%            0.20         6.80       0.00          5.20              4.80   
max            1.00        34.80       8.00         25.60             25.00   

       elo_diff  
count   3012.00  
mean       2.75  
std      138.16  
min     -440.46  
25%      -88.37  
50%        2.66  
75%       91.85  
max      484.91  


In [74]:
# Defines small function that builds a fresh scaler and logistic regression pipeline each time you call it.
def make_model():
    return Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression()),
    ])

# Defines the evaluation. Takes list of features and checks seasons 2019-2022.
def walk_forward(feature_cols, val_seasons=(2019, 2020, 2021, 2022)):
    accs, losses = [], []
    for val_season in val_seasons:

        # Games before that season (training data) and games in that season (validation data)
        tr = games[games["season"] < val_season]
        va = games[games["season"] == val_season]

        model = make_model()
        model.fit(tr[feature_cols], tr["home_win"])

        # Build and train a model on the training data, then predict probabilities and classes on the validation data.
        probs = model.predict_proba(va[feature_cols])[:, 1]
        preds = (probs > 0.5).astype(int)
        accs.append(accuracy_score(va["home_win"], preds))
        losses.append(log_loss(va["home_win"], probs))

    # Returns averages across the seasons
    return np.mean(accs), np.mean(losses)

# Dictionary of experiements
feature_sets = {
    "v1 (week 3)":           v1_cols,
    "v1 + offense/defense":  v1_cols + ["pts_for_diff", "pts_against_diff"],
    "v1 + Elo":              v1_cols + ["elo_diff"],
    "Elo only":              ["elo_diff"],
    "everything (v2)":       v2_cols,
}

for name, cols in feature_sets.items():
    acc, ll = walk_forward(cols)
    print(f"{name:24} accuracy {acc:.3f}   log loss {ll:.4f}")

v1 (week 3)              accuracy 0.621   log loss 0.6539
v1 + offense/defense     accuracy 0.618   log loss 0.6523
v1 + Elo                 accuracy 0.638   log loss 0.6395
Elo only                 accuracy 0.637   log loss 0.6399
everything (v2)          accuracy 0.645   log loss 0.6385


In [75]:
# Picks the features
best_cols = v2_cols   # change to the winner from Cell 8

# Split by season
train = games[games["season"] <= 2022]
test = games[games["season"] >= 2023]

# Builds and trains the model. .fit() returns trained pipeline.
final = make_model().fit(train[best_cols], train["home_win"])
# Probs and preds get the test-game probabilities and 0/1 predictions.
probs = final.predict_proba(test[best_cols])[:, 1]
preds = (probs > 0.5).astype(int)

# Train the week 3 model again on same data.
v1_model = make_model().fit(train[v1_cols], train["home_win"])
v1_probs = v1_model.predict_proba(test[v1_cols])[:, 1]
v1_preds = (v1_probs > 0.5).astype(int)

# Prints the picks
print("home-pick baseline:", round(test["home_win"].mean(), 3))
print("v1  accuracy:", round(accuracy_score(test["home_win"], v1_preds), 3),
      " log loss:", round(log_loss(test["home_win"], v1_probs), 4))
print("new accuracy:", round(accuracy_score(test["home_win"], preds), 3),
      " log loss:", round(log_loss(test["home_win"], probs), 4))

results = test.assign(pred=preds)
for season, grp in results.groupby("season"):
    print(season,
          "home-pick:", round(grp["home_win"].mean(), 3),
          "model:", round(accuracy_score(grp["home_win"], grp["pred"]), 3))

home-pick baseline: 0.549
v1  accuracy: 0.63  log loss: 0.639
new accuracy: 0.653  log loss: 0.6291
2023 home-pick: 0.565 model: 0.632
2024 home-pick: 0.547 model: 0.677
2025 home-pick: 0.533 model: 0.649


In [76]:
import os
import joblib

os.makedirs("../models", exist_ok=True)
joblib.dump({"model": final, "features": best_cols}, "../models/logreg_v2.joblib")

['../models/logreg_v2.joblib']